# Model 9: Future Prediction (Window=7, Horizon=1)
This notebook demonstrates how to use a trained model to predict future Bitcoin prices using a window of 7 days to predict the next day.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
# Load Bitcoin historical data
df = pd.read_csv('BTC_USD_2013-10-01_2021-05-18-CoinDesk.csv', parse_dates=['Date'], index_col=['Date'])
bitcoin_prices = pd.DataFrame(df['Closing Price (USD)']).rename(columns={'Closing Price (USD)': 'Price'})
prices = bitcoin_prices['Price'].to_numpy()

In [ ]:
# Utility function for plotting
def plot_time_series(timesteps, values, format='.', start=0, end=None, label=None):
    plt.plot(timesteps[start:end], values[start:end], format, label=label)
    plt.xlabel('Time')
    plt.ylabel('BTC Price')
    if label:
        plt.legend(fontsize=14)
    plt.grid(True)

In [ ]:
# Windowing functions
def get_labelled_windows(x, horizon=1):
    return x[:, :-horizon], x[:, -horizon:]
def make_windows(x, window_size=7, horizon=1):
    window_step = np.expand_dims(np.arange(window_size+horizon), axis=0)
    window_indexes = window_step + np.expand_dims(np.arange(len(x)-(window_size+horizon-1)), axis=0).T
    windowed_array = x[window_indexes]
    windows, labels = get_labelled_windows(windowed_array, horizon=horizon)
    return windows, labels
def make_train_test_splits(windows, labels, test_split=0.2):
    split_size = int(len(windows) * (1-test_split))
    train_windows = windows[:split_size]
    train_labels = labels[:split_size]
    test_windows = windows[split_size:]
    test_labels = labels[split_size:]
    return train_windows, test_windows, train_labels, test_labels

In [ ]:
# Prepare windowed data
HORIZON = 1
WINDOW_SIZE = 7
full_windows, full_labels = make_windows(prices, window_size=WINDOW_SIZE, horizon=HORIZON)
train_windows, test_windows, train_labels, test_labels = make_train_test_splits(full_windows, full_labels)

## Build and Train Model (Dense)

In [ ]:
from tensorflow.keras import layers
tf.random.set_seed(42)
model_9 = tf.keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(WINDOW_SIZE,)),
    layers.Dense(HORIZON)
])
model_9.compile(loss='mae', optimizer=tf.keras.optimizers.Adam())
model_9.fit(train_windows, train_labels, epochs=100, verbose=0, batch_size=128, validation_data=(test_windows, test_labels))

## Predict Future Prices

In [ ]:
def predict_future(model, last_window, n_future=30):
    future_preds = []
    current_window = last_window.copy()
    for _ in range(n_future):
        pred = model.predict(current_window.reshape(1, -1))[0][0]
        future_preds.append(pred)
        current_window = np.roll(current_window, -1)
        current_window[-1] = pred
    return np.array(future_preds)
last_window = prices[-WINDOW_SIZE:]
future_preds = predict_future(model_9, last_window, n_future=30)

## Visualize Future Predictions

In [ ]:
plt.figure(figsize=(10, 7))
plot_time_series(timesteps=bitcoin_prices.index[-100:], values=prices[-100:], label='Historical')
future_timesteps = pd.date_range(bitcoin_prices.index[-1], periods=31, freq='D')[1:]
plot_time_series(timesteps=future_timesteps, values=future_preds, format='-', label='Future Prediction')